An embedding is basically a list of numbers — a vector — that represents the *meaning* of a piece of text, not just the words themselves. So if I take a sentence like 'The cat sat on the mat' and run it through an embedding model, I get back something like 1500 numbers. The important part is: if I take a totally different sentence that means roughly the same thing, like 'A feline rested on the rug,' its vector will be numerically very close to the first one — even though barely any words match. But something unrelated, like 'The stock market crashed today,' will produce a vector that's numerically far away from both. So embeddings capture meaning, not just keywords.

Now, why do I even need this? Because once I've split my document into chunks, I can't rely on plain keyword search to find the right chunk for a user's question. If someone asks 'how do I reset my password,' a keyword search might completely miss a chunk that says 'to recover account access, follow these steps' — same meaning, different words. Embeddings let me compare meaning instead of exact wording, which is the entire backbone of RAG, Retrieval-Augmented Generation.

The general process looks like this: First, I chunk my document — which I've already covered using splitters like `RecursiveCharacterTextSplitter`. Second, I pick an embedding model — this is a dedicated AI model whose only job is converting text into a vector. Third, I pass every single chunk through that model and get back one vector per chunk. Fourth, I store each chunk's text alongside its vector inside a vector database, like Chroma or Pinecone, so I can search it later. Then, when a user actually asks a question, I embed their question too — using the *exact same* embedding model — and finally, I compare that question's vector against all my stored chunk vectors, and retrieve whichever ones are closest in meaning. Those become the context I feed into the LLM.

In terms of actual providers, LangChain itself doesn't have its own embedding model — it just gives me a common interface, and I plug in whichever provider I want. The most common ones are `OpenAIEmbeddings`, which is paid but very high quality, and `HuggingFaceEmbeddings`, which is free, open-source, and can even run locally on my own machine without hitting any API. There's also `CohereEmbeddings` and `GoogleGenerativeAIEmbeddings` as other commercial options, and `OllamaEmbeddings` if I want to run models fully offline.

Every embedding class in LangChain exposes two main methods: `embed_documents()`, for embedding a list of chunks at once, and `embed_query()`, for embedding a single piece of text, usually the user's question.

One critical rule I always keep in mind: I have to use the *exact same* embedding model for both storing my chunks and embedding my search query. If I embed my documents with OpenAI but query with HuggingFace, the two sets of vectors live in completely different numeric spaces, and the comparison becomes meaningless — the results would basically be garbage.

So, to summarize the whole flow in one line: chunking breaks my document into pieces, embedding converts each piece into a meaning-based vector using a model like OpenAIEmbeddings or HuggingFaceEmbeddings, and that vector is what eventually gets stored and searched in a vector database to power retrieval in a RAG pipeline."


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


/Users/shivamteli/langchaincourse/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18591.61it/s]


In [6]:
query_vector = embedding_model.embed_query("What is your name ?")
print(len(query_vector))
print(query_vector)

384
[-0.07434389740228653, 0.030666647478938103, 0.04120275378227234, 0.05998126417398453, -0.07701468467712402, -0.08020322024822235, 0.11509563028812408, -0.01655294932425022, 0.09429067373275757, -0.009441886097192764, -0.04425344243645668, -0.10292243957519531, 0.05429783836007118, 0.022460253909230232, -0.02801494114100933, -0.014137545600533485, 0.013136357069015503, -0.007768149953335524, -0.030992697924375534, -0.0916401818394661, -0.08531030267477036, 0.03048103302717209, -0.030657142400741577, -0.007428955286741257, -0.06741336733102798, -0.013540137559175491, -0.035039134323596954, 0.07582662254571915, -0.07940754294395447, -0.11881019175052643, -0.00984242744743824, -0.017222262918949127, 0.07728053629398346, 0.06858330219984055, 0.04789077118039131, -0.01802103780210018, -0.11063333600759506, 0.057727567851543427, 0.019396401941776276, 0.020351592451334, -0.0013002932537347078, -0.0673774778842926, -0.04487540200352669, 0.01898437738418579, 0.018741918727755547, -0.0120542

In [8]:
# how to embed multiple chunks 
chunks=[
    "Hi my name is Shivam",
    "I am in my third year of my undergraduate studies",
    "dont know why am here"
]
chunk_vectors = embedding_model.embed_documents(chunks)
print(len(chunk_vectors))
print(len(chunk_vectors[0]))
print(chunk_vectors[0])

3
384
[-0.011514448560774326, -0.0469062402844429, -0.03435901552438736, 0.05882927030324936, -0.07574637979269028, -0.05689218267798424, 0.09444095939397812, -0.01151785533875227, 0.054346274584531784, -0.044105201959609985, -0.053956978023052216, -0.1079077199101448, 0.013184089213609695, -0.0014365616953000426, 0.07315542548894882, 0.008210809901356697, 0.007727562915533781, 0.040692999958992004, -0.06041281297802925, -0.03479123115539551, -0.04480237513780594, -0.009139545261859894, -0.05038216710090637, -0.0057183681055903435, -0.11782930046319962, -0.03087514452636242, 0.010293065570294857, 0.10726739466190338, 0.0325704962015152, -0.02323431521654129, 0.07299615442752838, -0.0052216448821127415, 0.04047422483563423, 0.08360382169485092, -0.023776382207870483, 0.053457871079444885, -0.12745580077171326, -0.06369347870349884, -0.05224943161010742, 0.013208792544901371, 0.0011818206403404474, 0.009133918210864067, -0.04623181372880936, -0.02962062507867813, 0.009288369677960873, -0